# Notebook 01: Datasets
Builds every dataset that does not depend on a trained model:

1. **Benchmarks:** GSM8K train (training source) and GSM-Symbolic (evaluation: a seeded sample of
   `SYMBOLIC_PER_TEMPLATE` instances from each of the 100 templates, 1,500 questions by default)
2. **Own traces:** the base Qwen3-8B (4-bit, greedy, thinking mode) writes its own trace $R$ and answer $A$
   for every GSM8K and GSM-Symbolic question
3. **Filtering:** keep traces that finish within the token limit, do not loop and reach the gold answer
4. **Training sets:** one file per adapter role, all from the same GSM8K samples
5. **Trap candidates:** DeepSeek-V4.1-Flash inserts one error into the base model's own trace for a
   template-balanced subset of the GSM-Symbolic questions it solved; automatic checks

The training data is self-distilled: every adapter learns from the base model's own traces and answers,
not from a teacher's. The adapters therefore learn their role (write a trace, or answer from one) without
also learning a new reasoning or answering style, and the traps edit traces written in the style the
models will see. Problems the base model gets wrong, or where it loops, are dropped.

The known-correct intersection and the manual review need the trained conditions, so they are in
`03_evaluation.ipynb`.

**Outputs**
```
data/raw/gsm8k_train.json, gsm_symbolic.json
data/traces/own_gsm8k.jsonl, own_symbolic.jsonl    every generation (resumable)
data/traces/gsm8k_valid.json, symbolic_valid.json  generations that passed the filter
data/traces/filter_stats.json
data/sft/{monolithic, thinker, soft_solver, hard_solver}.json, dataset_summary.json
data/trap/trap_selection.json, trap_candidates.jsonl
```
Step 2 needs the GPU and is the longest step of the study: about 8,000 generations of up to
`MAX_NEW_TOKENS_TRACE` tokens each, which takes days on a 4070. Every generation is appended to a JSONL file as
it is made, so the cell can be stopped and re-run at any time.

In [ ]:
import sys
sys.path.insert(0, "../src")

from faithfulness import config

import pandas as pd
from faithfulness.data import (load_gsm8k_train, load_gsm_symbolic, load_json, load_jsonl, sample_per_template,
                               save_json)

## 1. Benchmarks

In [ ]:
gsm8k = load_gsm8k_train()
save_json(gsm8k, config.GSM8K_TRAIN_PATH)
print(f"GSM8K train: {len(gsm8k)} problems -> {config.GSM8K_TRAIN_PATH}")

symbolic = sample_per_template(load_gsm_symbolic())
save_json(symbolic, config.GSM_SYMBOLIC_PATH)
n_templates = len({p["template_id"] for p in symbolic})
print(f"GSM-Symbolic: {len(symbolic)} questions ({config.SYMBOLIC_PER_TEMPLATE} per template, seed {config.SEED}) "
      f"from {n_templates} templates -> {config.GSM_SYMBOLIC_PATH}")

assert all(p["target_answer"] for p in gsm8k + symbolic), "A gold answer failed to parse"
print("\nExample GSM-Symbolic question:\n", symbolic[0]["question"], "\nTarget:", symbolic[0]["target_answer"])

## 2. Own traces
The untrained base model answers every question once, greedily, with the `reasoning` prompt (no system prompt,
Qwen3's native thinking format). Trace and answer must both finish within `MAX_NEW_TOKENS_TRACE` tokens.

Set `PILOT_LIMIT` to a small number first to check speed and filter rates.

In [ ]:
from faithfulness.evaluation import run_own_traces

PILOT_LIMIT = None   # None = every question

for name, problems in [("gsm8k", gsm8k), ("symbolic", symbolic)]:
    problems = problems if PILOT_LIMIT is None else problems[:PILOT_LIMIT]
    run_own_traces(problems, config.TRACE_DIR / f"own_{name}.jsonl")

## 3. Filtering
A generation is kept only if:
- it closes `</think>` and finishes its answer within the token limit (an unfinished trace is almost always a
  reasoning loop under greedy decoding)
- the answer matches the gold answer
- training pool only: the trace does not repeat the same sentence `LOOP_MIN_REPEATS` times (loops that
  happened to finish), so the adapters do not learn to loop

GSM8K survivors are the training pool. GSM-Symbolic survivors are the trap pool: every trap starts from a
finished trace with which the base model reached the correct answer.

In [ ]:
from faithfulness.data import filter_own_traces

valid, filter_stats = {}, {}
for name, path in [("gsm8k", config.GSM8K_VALID_PATH), ("symbolic", config.SYMBOLIC_VALID_PATH)]:
    valid[name], filter_stats[name] = filter_own_traces(load_jsonl(config.TRACE_DIR / f"own_{name}.jsonl"),
                                                        check_loops=(name == "gsm8k"))
    save_json(valid[name], path)
save_json({**filter_stats, "max_new_tokens_trace": config.MAX_NEW_TOKENS_TRACE,
           "loop_min_repeats": config.LOOP_MIN_REPEATS, "loop_min_chars": config.LOOP_MIN_CHARS},
          config.TRACE_DIR / "filter_stats.json")

pd.DataFrame(filter_stats).fillna(0).astype(int)

In [ ]:
pd.DataFrame({name: pd.Series([v["generated_tokens"] for v in rows]).describe()
              for name, rows in valid.items()}).round(0).T

## 4. Training sets
| File | Role | Input | Target |
| :--- | :--- | :--- | :--- |
| `monolithic.json` | Monolithic | $Q$ | $(R, A)$ |
| `thinker.json` | Thinker (both modular conditions) | $Q$ | $R$ |
| `soft_solver.json` | Soft Modular Solver | $(Q, R)$ | $A$ |
| `hard_solver.json` | Hard Bottleneck Solver | $R$ | $A$ |

$R$ and $A$ are the base model's own trace and answer. The four sets use the same questions, traces and
answers: a sample is dropped from all of them if its longest example (Monolithic or Soft Solver) is over
`MAX_SEQ_LENGTH`. Exact prompt text lives in `src/faithfulness/prompts.py`.

In [ ]:
from transformers import AutoTokenizer
from faithfulness.data import build_training_sets

tokenizer = AutoTokenizer.from_pretrained(config.BASE_MODEL)
sft_sets, length_stats = build_training_sets(valid["gsm8k"], tokenizer)
for role, examples in sft_sets.items():
    save_json(examples, config.SFT_DIR / f"{role}.json")
print(length_stats)

# Sanity checks on the experimental contrast
by_id = {v["id"]: v for v in valid["gsm8k"]}
quoted = 0
for soft, hard in zip(sft_sets["soft_solver"], sft_sets["hard_solver"]):
    item = by_id[soft["id"]]
    assert soft["completion"] == hard["completion"]
    assert item["question"] in soft["prompt"]
    # The Hard Solver input is R only: with R removed, no trace of Q may remain
    assert item["question"] not in hard["prompt"].replace(item["trace"], "")
    quoted += item["question"] in item["trace"]
print("Solver sets: identical targets, question passed only to the Soft Solver")

# A trace can restate the problem, in which case the Hard Solver sees Q through R (report this)
n = len(sft_sets["hard_solver"])
print(f"Traces quoting the full question verbatim: {quoted} / {n} ({quoted / n:.1%})")

In [ ]:
def count(text):
    return len(tokenizer(text, add_special_tokens=False).input_ids)

summary = {}
for role, examples in sft_sets.items():
    prompt_tokens = [count(e["prompt"]) for e in examples]
    completion_tokens = [count(e["completion"]) for e in examples]
    summary[role] = {
        "examples": len(examples),
        "prompt_tokens": sum(prompt_tokens),
        "completion_tokens (loss)": sum(completion_tokens),
        "max_example_tokens": max(p + c for p, c in zip(prompt_tokens, completion_tokens)),
    }
save_json(summary, config.SFT_DIR / "dataset_summary.json")
pd.DataFrame(summary).T

## 5. Trap candidates
**Selection** (seeded, reported): the valid GSM-Symbolic problems are grouped by template, templates are
shuffled, and one instance is taken from each template in turn, until `TRAP_CANDIDATES` problems are chosen.
This spreads the traps across templates.

**Trapped traces:** for each, DeepSeek changes one arithmetic or logical step of the base model's own
trace and rewrites the later steps to stay consistent with it (prompt: `TRAP_SYSTEM_PROMPT` in
`src/faithfulness/data.py`). Responses are appended to `trap_candidates.jsonl`, so the cell can be stopped and
re-run; failed calls are retried.

In [ ]:
from faithfulness.data import run_trap_generation, select_trap_candidates

selection = select_trap_candidates(valid["symbolic"])
save_json(selection, config.TRAP_DIR / "trap_selection.json")
sym_by_id = {v["id"]: v for v in valid["symbolic"]}
print(f"{len(selection)} candidates from {len({sym_by_id[i]['template_id'] for i in selection})} templates")

trap_inputs = [{"id": pid, "template_id": sym_by_id[pid]["template_id"], "question": sym_by_id[pid]["question"],
                "target_answer": sym_by_id[pid]["target_answer"], "clean_trace": sym_by_id[pid]["trace"]}
               for pid in selection]
run_trap_generation(trap_inputs, config.TRAP_DIR / "trap_candidates.jsonl")

In [ ]:
from faithfulness.data import auto_check_trap

traps = [t for t in load_jsonl(config.TRAP_DIR / "trap_candidates.jsonl") if not t["error"]]
problems = [p for t in traps for p in auto_check_trap(t)]
print(f"Traps generated: {len({t['id'] for t in traps})} / {len(selection)} | "
      f"passed automatic checks: {sum(not auto_check_trap(t) for t in traps)}")
print(pd.Series(problems, dtype=str).value_counts().rename("automatic check failures").to_string())
print("\nNext: 02_training.ipynb")